[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E8%B3%87%E6%96%99%E6%95%B4%E7%90%86%E5%92%8C%E4%BD%BF%E7%94%A8%E5%89%8D%E6%BA%96%E5%82%99/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E9%AB%92%E4%BA%82%E7%9A%84%E6%9C%83%E5%93%A1%E5%95%8F%E5%8D%B7%E5%A4%A7%E6%8E%83%E9%99%A4.ipynb)

# 🧹 茶語時光：髒亂的會員問卷大掃除

> 🎯 **情境**：行銷部用 Google 表單做了會員滿意度問卷，想知道「哪家店滿意度最高？」、「各地區客人喜歡幾分糖？」。但表單是開放填寫的：有人重複送出、性別寫 F 或 male、年齡寫「二十」或 999、門市寫「信義」或「西門町店」、甜度有「全糖」、「10分」、「正常」三種寫法……直接分析一定會錯。你的任務是**先把資料洗乾淨，再回答問題**。

| 素材檔案 | 內容 |
| --- | --- |
| `會員問卷_原始.csv` | 從 Google 表單匯出的 318 份會員問卷，未經任何整理（裡面有很多陷阱！） |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | 健康檢查——先看看資料有多髒 |
| ⬜ 任務 2 | 去除重複——同一個人只算一次 |
| ⬜ 任務 3 | 統一類別——一家店只能有一種寫法 |
| ⬜ 任務 4 | 救回數字欄位——年齡、購買次數、滿意度 |
| ⬜ 任務 5 | 整理 Email——大小寫、空白、全形符號 |
| ⬜ 任務 6 | 洗乾淨了！終於可以回答行銷部的問題 |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「老闆想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '會員問卷_原始.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E8%B3%87%E6%96%99%E6%95%B4%E7%90%86%E5%92%8C%E4%BD%BF%E7%94%A8%E5%89%8D%E6%BA%96%E5%82%99/%E8%B3%87%E6%96%99%E6%95%B4%E7%90%86%E5%92%8C%E4%BD%BF%E7%94%A8%E5%89%8D%E6%BA%96%E5%82%99_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['會員問卷_原始.csv']


## 任務 1：健康檢查——先看看資料有多髒

**白話說明**：分析前先做「健康檢查」：多少筆、每欄的型態、缺值、重複，以及類別欄位到底有幾種寫法。

In [2]:
import pandas as pd

raw = pd.read_csv('會員問卷_原始.csv', dtype=str, keep_default_na=False)   # 先全部當文字讀，避免 pandas 自作主張轉型
print('筆數與欄數：', raw.shape)
print('完全重複的列：', raw.duplicated().sum())
print('重複的會員編號：', raw['會員編號'].duplicated().sum())
print('\n性別有幾種寫法：', raw['性別'].unique())
print('\n甜度有幾種寫法：', raw['甜度'].nunique(), '→', raw['甜度'].unique())
print('\n門市有幾種寫法：', raw['常去門市'].nunique())
print('\n年齡的怪值：', [v for v in raw['年齡'].unique() if not v.isdigit()][:10])

筆數與欄數： (318, 12)
完全重複的列： 12
重複的會員編號： 21

性別有幾種寫法： <StringArray>
['女', '男生', 'M', '', '男', 'male', '男 ', 'F', ' 女', 'female', '女生']
Length: 11, dtype: str

甜度有幾種寫法： 17 → <StringArray>
['正常糖',  '半糖', '五分糖',  '少糖',  '3分', '七分糖', '三分糖', '10分',  '0分',  '7分', '50%',
  '微糖',  '全糖',  '無糖',  '5分',  '正常',  '去糖']
Length: 17, dtype: str

門市有幾種寫法： 21

年齡的怪值： ['23歲', '二十', '33歲', '', '26 ', '31 ', '29歲', '34 ', '-1', '24歲']


> 💡 **看懂結果**：只有 5 家門市，卻有二十幾種寫法；甜度只有 5 種，卻寫出十幾種。**如果直接用 groupby 統計，「信義店」和「台北信義店」會被當成兩家店！**
>
> 🤖 **換你問 AI**：「請幫我對這份問卷做資料健康檢查：列出每欄的缺值數、重複列數，以及每個類別欄位的所有不同寫法。」

## 任務 2：去除重複——同一個人只算一次

**兩種重複**：① 按了兩次送出（整列一模一樣）；② 同一位會員後來改了答案又送一次——這種要**保留最新的一筆**。

In [3]:
df = raw.drop_duplicates().copy()                                      # ① 刪除完全相同的列
print('刪除完全重複後：', len(raw), '→', len(df))

df['填答時間'] = pd.to_datetime(df['填答時間'], format='%Y/%m/%d %H:%M')
df = df.sort_values('填答時間').drop_duplicates('會員編號', keep='last')   # ② 依時間排序，同一會員只留最後一筆
print('同一會員只留最新一筆後：', len(df))

刪除完全重複後： 318 → 306
同一會員只留最新一筆後： 297


## 任務 3：統一類別——一家店只能有一種寫法

**白話說明**：先 `str.strip()` 去掉前後空白，再用「對照表」（字典）把各種寫法換成標準名稱。

In [4]:
gender_map = {'女': '女', 'F': '女', 'female': '女', '女生': '女', '男': '男', 'M': '男', 'male': '男', '男生': '男'}
df['性別'] = df['性別'].str.strip().map(gender_map)                     # 對照表裡沒有的（空白）會變成 NaN

store_map = {'信義店': '台北信義店', '信義': '台北信義店', '臺北信義店': '台北信義店',
             '西門店': '台北西門店', '西門町店': '台北西門店', '臺北西門店': '台北西門店',
             '逢甲店': '台中逢甲店', '逢甲': '台中逢甲店', '臺中逢甲店': '台中逢甲店',
             '成大店': '台南成大店', '臺南成大店': '台南成大店', '台南成大': '台南成大店',
             '巨蛋店': '高雄巨蛋店', '高雄巨蛋': '高雄巨蛋店'}
df['常去門市'] = df['常去門市'].str.strip().replace(store_map)          # replace：不在對照表的值保持原樣

sweet_map = {'全糖': '正常糖', '10分': '正常糖', '正常': '正常糖', '7分': '少糖', '七分糖': '少糖',
             '5分': '半糖', '50%': '半糖', '五分糖': '半糖', '3分': '微糖', '三分糖': '微糖', '0分': '無糖', '去糖': '無糖'}
df['甜度'] = df['甜度'].str.strip().replace(sweet_map)

drink_map = {'珍奶': '珍珠奶茶', '波霸奶茶': '珍珠奶茶', '四季春': '四季春青茶', '鮮奶綠茶': '鮮奶綠', '養樂多綠': '多多綠',
             '黑糖鮮奶': '黑糖珍珠鮮奶', '黑糖珠珠鮮奶': '黑糖珍珠鮮奶', '芋泥鮮奶': '芋頭鮮奶'}
df['最愛飲品'] = df['最愛飲品'].str.strip().replace(drink_map)

for col in ['性別', '常去門市', '甜度', '最愛飲品']:
    print(col, '→', sorted(df[col].dropna().unique()))

性別 → ['女', '男']
常去門市 → ['台中逢甲店', '台北信義店', '台北西門店', '台南成大店', '高雄巨蛋店']
甜度 → ['半糖', '少糖', '微糖', '正常糖', '無糖']
最愛飲品 → ['冬瓜檸檬', '四季春青茶', '多多綠', '珍珠奶茶', '芋頭鮮奶', '鮮奶綠', '黑糖珍珠鮮奶']


> 🤖 **換你問 AI**：「門市欄位有這些寫法（貼上 unique() 結果），請幫我建立一個對照表字典，統一成 5 家標準門市名稱。」

## 任務 4：救回數字欄位——年齡、購買次數、滿意度

**白話說明**：
- 年齡：從「25歲」中抽出數字；「二十」、「999」、「-1」這種**不合理的值改成缺值（NaN）**，而不是亂猜。
- 購買次數：「三次」→ 3、「每天」→ 7。
- 滿意度：只接受 1～5，其他（0、99）視為無效。

In [5]:
age = pd.to_numeric(df['年齡'].str.extract(r'(\d+)')[0], errors='coerce')   # 抽出數字；抽不到的變 NaN
df['年齡'] = age.where(age.between(12, 90))                                   # 不在 12～90 歲之間的視為無效

cn = {'一': '1', '二': '2', '三': '3', '四': '4', '五': '5', '六': '6', '七': '7'}
times = df['每週購買次數'].str.replace('次', '').replace({'每天': '7', **cn})
df['每週購買次數'] = pd.to_numeric(times, errors='coerce')

sat = pd.to_numeric(df['滿意度'], errors='coerce')
df['滿意度'] = sat.where(sat.between(1, 5))

display(df[['年齡', '每週購買次數', '滿意度']].describe().round(2))
print('\n各欄缺值數：\n', df[['性別', '年齡', '每週購買次數', '滿意度']].isnull().sum())

,年齡,每週購買次數,滿意度
count,277.00,283.00,282.00
mean,29.99,2.60,3.95
std,7.08,1.76,0.78
min,16.00,1.00,2.00
25%,25.00,1.00,3.00
50%,30.00,2.00,4.00
75%,35.00,3.00,4.75
max,54.00,7.00,5.00



各欄缺值數：
 性別        12
年齡        20
每週購買次數    14
滿意度       15
dtype: int64


## 任務 5：整理 Email——大小寫、空白、全形符號

In [6]:
email = (df['Email'].str.strip()                     # 去前後空白
                    .str.lower()                      # 全部轉小寫
                    .str.replace('＠', '@'))          # 全形＠ → 半形 @
df['Email'] = email.where(email.str.contains(r'^[\w.]+@[\w.]+\.\w+$'))   # 格式不對的（空白、none）變 NaN
print('有效 Email：', df['Email'].notnull().sum(), '／', len(df))

有效 Email： 283 ／ 297


## 任務 6：洗乾淨了！終於可以回答行銷部的問題

In [7]:
print('▶ 各門市平均滿意度：')
display(df.groupby('常去門市')['滿意度'].agg(['count', 'mean']).round(2).sort_values('mean', ascending=False))

print('\n▶ 各門市甜度偏好（%）：')
sweet_order = ['正常糖', '少糖', '半糖', '微糖', '無糖']
display((pd.crosstab(df['常去門市'], df['甜度'], normalize='index')[sweet_order] * 100).round(0))

df.to_csv('會員問卷_清理後.csv', index=False, encoding='utf-8-sig')
print('\n✅ 已存成 會員問卷_清理後.csv，共', len(df), '筆')

▶ 各門市平均滿意度：


,count,mean
常去門市,,
台南成大店,55,4.29
台北信義店,56,4.16
台中逢甲店,63,4.11
高雄巨蛋店,53,3.75
台北西門店,55,3.42



▶ 各門市甜度偏好（%）：


甜度,正常糖,少糖,半糖,微糖,無糖
常去門市,,,,,
台中逢甲店,19.0,27.0,17.0,22.0,16.0
台北信義店,13.0,23.0,21.0,23.0,20.0
台北西門店,31.0,14.0,34.0,9.0,12.0
台南成大店,43.0,19.0,12.0,19.0,7.0
高雄巨蛋店,18.0,29.0,38.0,11.0,5.0



✅ 已存成 會員問卷_清理後.csv，共 297 筆


> 💡 **看懂結果**：
> - 台南成大店選「正常糖」的比例明顯最高，台北信義店（上班族）選正常糖的比例最低，微糖加無糖超過四成——**「台南人愛甜」在我們的資料裡是真的**。健康系列新品可以先在信義店試賣。
> - 回頭想想：如果沒有統一甜度寫法，「全糖」、「10分」、「正常」會被拆成三組，台南的正常糖比例會被嚴重低估。**洗資料不是雜事，它直接決定結論對不對。**
>
> 🤖 **換你問 AI**：「請把整理後的問卷依門市計算平均滿意度，並做出門市 × 甜度的百分比交叉表，用白話說明各地區的甜度偏好差異。」

---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [8]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：年齡缺值不要隨便補：用「各門市年齡的中位數」補上

<details><summary>💡 卡住了？點我看提示</summary>

`groupby('常去門市')['年齡'].transform('median')` 會回傳和原表一樣長的欄位，每一列都是「自己門市」的中位數，剛好拿來 `fillna`。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
store_median = df.groupby(___)['年齡'].transform('median')
df['年齡'] = df['年齡'].fillna(___)
print('補值後年齡缺值：', df['年齡'].isnull().sum())

In [ ]:
# @title ✅ 執行這格，檢查練習 1
check('store_median', df.groupby('常去門市')['年齡'].transform('median'))

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
store_median = df.groupby('常去門市')['年齡'].transform('median')
df['年齡'] = df['年齡'].fillna(store_median)
print('補值後年齡缺值：', df['年齡'].isnull().sum())
```

</details>

### ✍️ 練習 2：顧客建議裡，有幾則提到「甜」？幾則提到「外送」？

<details><summary>💡 卡住了？點我看提示</summary>

`df['建議'].str.contains('甜')` 會得到 True/False，`.sum()` 就是 True 的數量。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
sweet_n = df['建議'].str.___('甜').sum()
delivery_n = df['建議'].str.contains(___).sum()
print('提到甜：', sweet_n, '則；提到外送：', delivery_n, '則')

In [ ]:
# @title ✅ 執行這格，檢查練習 2
check('sweet_n', df['建議'].str.contains('甜').sum())
check('delivery_n', df['建議'].str.contains('外送').sum())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
sweet_n = df['建議'].str.contains('甜').sum()
delivery_n = df['建議'].str.contains('外送').sum()
print('提到甜：', sweet_n, '則；提到外送：', delivery_n, '則')
```

</details>

### ✍️ 練習 3：清理前後，「台南成大店」的筆數差多少？為什麼？

<details><summary>💡 卡住了？點我看提示</summary>

`(表格['欄位'] == 值).sum()` 可以數出符合的筆數。清理後多了「被統一回來的各種寫法」，少了「重複填答」。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
before = (raw['常去門市'] == ___).sum()             # 清理前，剛好寫成「台南成大店」的筆數
after = (df[___] == '台南成大店').sum()              # 清理後
print('清理前：', before, '清理後：', after)

In [ ]:
# @title ✅ 執行這格，檢查練習 3
check('before', (raw['常去門市'] == '台南成大店').sum())
check('after', (df['常去門市'] == '台南成大店').sum())

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
before = (raw['常去門市'] == '台南成大店').sum()
after = (df['常去門市'] == '台南成大店').sum()
print('清理前：', before, '清理後：', after)
# 各種寫法（台南店、成大店……）被統一回來，筆數變多；但重複填答被刪掉，又會少一些
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：健康檢查——先看看資料有多髒
- ✅ 任務 2：去除重複——同一個人只算一次
- ✅ 任務 3：統一類別——一家店只能有一種寫法
- ✅ 任務 4：救回數字欄位——年齡、購買次數、滿意度
- ✅ 任務 5：整理 Email——大小寫、空白、全形符號
- ✅ 任務 6：洗乾淨了！終於可以回答行銷部的問題

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以交給老闆的報告。